# miRNA Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/miRNA/`

miRNA-specific notes vs RNA/CNV:
- Small feature space (~1000 miRNAs) — same scale as proteins
- Read counts need log2(RPM+1) normalization (same as RNA)
- No z-score before filtering — applied per-dataset after selection
- TARGET_BROAD = 200 (same as proteins, feature space too small for 1000)

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 percentile |
| 2 | conservative | Variance > 97.5 percentile |
| 3 | standard | Variance > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top 200 by composite |
| 5 | balanced | Var top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/miRNA/mirna_preprocessing.ipynb
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.mirna.tsv").exists()
)

MIRNA_FILE   = BASE_DIR / "data" / "TCGA-BRCA.mirna.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "miRNA" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "01_Causal_feature_extraction" / "miRNA" / "mirna_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_MIRNA    = 50
TARGET_BROAD = 200   # miRNA space is small (~1000); 1000 not feasible

assert MIRNA_FILE.exists(),   f"Not found: {MIRNA_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base        : {BASE_DIR}")
print(f"Output      : {OUTPUT_DIR}")
print(f"Cache       : {STATS_CACHE}")
print(f"TARGET_BROAD: {TARGET_BROAD}")
print("Paths OK")


Base        : C:\Users\olegk\Desktop\Thesis_v3
Output      : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA\statistical_filtered
Cache       : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA\mirna_statistics_cache.csv
TARGET_BROAD: 200
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Peeking at file structure...")
peek = pd.read_csv(MIRNA_FILE, sep="\t", index_col=0, nrows=5)
print(f"  Shape (5 rows): {peek.shape}  (miRNAs x samples)")
print(f"  Index name    : {peek.index.name}")
print(f"  First miRNAs  : {peek.index.tolist()}")
print(f"  First samples : {peek.columns[:3].tolist()}")
print(f"  Value range   : [{peek.values.flatten()[~np.isnan(peek.values.flatten())].min():.2f}, "
      f"{peek.values.flatten()[~np.isnan(peek.values.flatten())].max():.2f}]")
print()
print("Check: are values raw counts, RPM, or already log-transformed?")
print("  If max > 1000 -> raw counts or RPM, need log2(x+1)")
print("  If max < 20   -> likely already log-transformed")


Peeking at file structure...
  Shape (5 rows): (5, 1202)  (miRNAs x samples)
  Index name    : miRNA_ID
  First miRNAs  : ['hsa-let-7a-1', 'hsa-let-7a-2', 'hsa-let-7a-3', 'hsa-let-7b', 'hsa-let-7c']
  First samples : ['TCGA-D8-A146-01A', 'TCGA-AQ-A0Y5-01A', 'TCGA-C8-A1HI-01A']
  Value range   : [6.68, 17.94]

Check: are values raw counts, RPM, or already log-transformed?
  If max > 1000 -> raw counts or RPM, need log2(x+1)
  If max < 20   -> likely already log-transformed


In [5]:
print("Loading miRNA data...")
mirna_raw = pd.read_csv(MIRNA_FILE, sep="\t", index_col=0)
print(f"  Raw shape : {mirna_raw.shape}  (miRNAs x samples)")

# Transpose to samples x miRNAs
mirna = mirna_raw.T.copy()
print(f"  Transposed: {mirna.shape}  (samples x miRNAs)")

print("Loading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples  : {len(outcome)}")
print(f"  OS.time  : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events   : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(mirna.index) & set(outcome.index))
mirna   = mirna.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap  : {len(common)} samples")


Loading miRNA data...
  Raw shape : (1881, 1202)  (miRNAs x samples)
  Transposed: (1202, 1881)  (samples x miRNAs)
Loading outcome...
  Samples  : 1076
  OS.time  : 1 - 8605 days
  Events   : 150 (13.9%)
  Overlap  : 1056 samples


In [6]:
print("Quality control...")

# Remove miRNAs with >20% missing
miss_frac = mirna.isna().mean(axis=0)
mirna = mirna.loc[:, miss_frac <= 0.20]
print(f"  After missing filter (>20%)  : {mirna.shape[1]:,} miRNAs")

# Impute residual missing with median
imp   = SimpleImputer(strategy="median")
mirna = pd.DataFrame(imp.fit_transform(mirna), index=mirna.index, columns=mirna.columns)

# Remove zero-variance miRNAs
var   = mirna.var(axis=0)
mirna = mirna.loc[:, var > 0]
print(f"  After zero-variance removal  : {mirna.shape[1]:,} miRNAs")

# Normalization: log2(x+1) if values suggest raw/RPM counts
max_val = mirna.values.max()
if max_val > 20:
    print(f"  Max value {max_val:.1f} > 20 -> applying log2(x+1)")
    mirna = np.log2(mirna + 1)
else:
    print(f"  Max value {max_val:.1f} <= 20 -> data appears pre-transformed, skipping log2")

print(f"  After normalization: mean={mirna.mean().mean():.4f}  std={mirna.std().mean():.4f}")


Quality control...
  After missing filter (>20%)  : 1,881 miRNAs
  After zero-variance removal  : 1,601 miRNAs
  Max value 19.4 <= 20 -> data appears pre-transformed, skipping log2
  After normalization: mean=1.3318  std=0.4350


In [7]:
print("Variance pre-filter: keep top 75%...")
var    = mirna.var(axis=0)
mirna_var = mirna.loc[:, var > var.quantile(0.25)]
print(f"  Before : {mirna.shape[1]:,}  After : {mirna_var.shape[1]:,} miRNAs")


Variance pre-filter: keep top 75%...
  Before : 1,601  After : 1,200 miRNAs


In [8]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} miRNAs")
else:
    import time
    os_time  = outcome["OS.time"].values
    mirnas   = mirna_var.columns.tolist()
    n        = len(mirnas)
    print(f"Computing statistics for {n:,} miRNAs...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for m in mirnas:
        r, p = spearmanr(mirna_var[m].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(p) else float(p))
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")
    print(f"    Done in {time.time()-t0:.1f}s")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        mirna_var.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation, consistent with all modalities)
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for m in mirnas:
        x_rank = rankdata(mirna_var[m].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "mirna":         mirnas,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      mirna_var.var().values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05  : {n_fdr:,} / {len(stats):,} miRNAs")
print(f"  Top 10 by composite score:")
print(stats[["mirna","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))


Computing statistics for 1,200 miRNAs...
  [1/3] Spearman correlation...
    Done in 1.7s
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to mirna_statistics_cache.csv
  FDR < 0.05  : 96 / 1,200 miRNAs
  Top 10 by composite score:
         mirna  abs_spearman  mutual_info  distance_corr  composite
   hsa-mir-99a      0.197277     0.041143       0.197277   7.333333
   hsa-mir-202      0.138635     0.042139       0.138635  16.000000
   hsa-mir-326      0.126384     0.058527       0.126384  16.333333
 hsa-mir-101-1      0.196372     0.030926       0.196372  18.333333
  hsa-mir-374c      0.169401     0.034121       0.169401  18.666667
   hsa-mir-379      0.135202     0.040690       0.135202  19.000000
 hsa-mir-101-2      0.195217     0.030027       0.195217  20.666667
hsa-mir-125b-2      0.122166     0.044498       0.122166  23.333333
  hsa-mir-1258      0.118229     0.042275       0.118229  28.000000
   hsa-mir-134      0.118822     0.041647    

In [9]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")
print(f"Note: miRNA space is small ({len(stats)} miRNAs); TARGET_BROAD={TARGET_BROAD}")

scaler = StandardScaler()

def build(mirna_list, min_m=MIN_MIRNA):
    """Select miRNAs; pad with top composite miRNAs if below minimum.
    Z-score applied per-dataset after selection."""
    mirna_list = [m for m in mirna_list if m in mirna_var.columns]
    if len(mirna_list) < min_m:
        have = set(mirna_list)
        mirna_list += [m for m in stats["mirna"] if m not in have][:min_m - len(mirna_list)]
    subset = mirna_var[mirna_list]
    normed = pd.DataFrame(
        scaler.fit_transform(subset),
        index=subset.index, columns=subset.columns
    )
    return normed

created = []

# V1 Ultra-Conservative: variance > 98.3 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["mirna"].tolist())
fname = f"mirna_1_ultra_conservative_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Variance > 98.3 pct","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} miRNAs")

# V2 Conservative: variance > 97.5 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["mirna"].tolist())
fname = f"mirna_2_conservative_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Variance > 97.5 pct","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} miRNAs")

# V3 Standard: variance > 95.9 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["mirna"].tolist())
fname = f"mirna_3_standard_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Variance > 95.9 pct","file":fname})
print(f"  V3 standard           : {len(df.columns):,} miRNAs")

# V4 FDR-significant: top TARGET_BROAD by composite
fdr_mirnas = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["mirna"].tolist()
df    = build(fdr_mirnas, min_m=TARGET_BROAD)
fname = f"mirna_4_fdr_significant_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":f"FDR<0.05, top {TARGET_BROAD} composite fallback","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} miRNAs  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: top 25% variance then top 10% Spearman
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["mirna"].tolist())
fname = f"mirna_5_balanced_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Var top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} miRNAs")

# V6 Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["mirna"].tolist())
fname = f"mirna_6_correlation_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} miRNAs")

# V7 Top Correlated: top 100 pos + top 100 neg
top_pos = stats[stats["spearman_rho"] > 0].head(100)["mirna"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["mirna"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["mirna"].isin(set(top_pos + top_neg))][
    ["mirna","spearman_rho","pval","fdr","variance"]
].to_csv(OUTPUT_DIR / "mirna_7_top_correlated_annotated.csv", index=False)
fname = f"mirna_7_top_correlated_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} miRNAs")

# V8 Composite: top TARGET_BROAD by average rank
df    = build(stats.head(TARGET_BROAD)["mirna"].tolist(), min_m=TARGET_BROAD)
fname = f"mirna_8_composite_{len(df.columns)}mirnas.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":f"Avg rank Spearman+MI+Dcor, top {TARGET_BROAD}","file":fname})
print(f"  V8 composite          : {len(df.columns):,} miRNAs")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "mirna_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA\statistical_filtered
Note: miRNA space is small (1200 miRNAs); TARGET_BROAD=200
  V1 ultra_conservative : 50 miRNAs
  V2 conservative       : 50 miRNAs
  V3 standard           : 50 miRNAs
  V4 fdr_significant    : 200 miRNAs  (raw FDR<0.05: 96)
  V5 balanced           : 50 miRNAs
  V6 correlation        : 50 miRNAs
  V7 top_correlated     : 200 miRNAs
  V8 composite          : 200 miRNAs

 V               name   n                                logic
V1 ultra_conservative  50                  Variance > 98.3 pct
V2       conservative  50                  Variance > 97.5 pct
V3           standard  50                  Variance > 95.9 pct
V4    fdr_significant 200 FDR<0.05, top 200 composite fallback
V5           balanced  50      Var top 25% -> Spearman top 10%
V6        correlation  50              Abs Spearman > 97.5 pct
V7     top_correlated 200   Top 100 pos + top 100 neg Spearm